# Train balance bot using PPO with domain randomization

Run each cell by pressing `shift + enter`.

We redo the original training and add more phases to our curriculum learning. In addition to training the bot to stay upright (phase 1) and reduce movement off the origin (phase 2), we introduce increasingly difficult domain randomization schemes in phases 3, 4 and 5.

In [1]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

In [2]:
# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import the custom environment and PPO training module
from envs.balance_bot_env_dr import BalanceBotEnv, DomainRandomConfig
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [3]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 4               # Number of parallel environments. Only the first will be rendered.
STEPS_PER_ENV = 500_000    # Number of simulation steps to perform per environment

## Configure PPO and environment

In [4]:
# Configure PPO
ppo_config = PPOConfig(
    exp_name = "balance-bot-ppo",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 16,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 16,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Number of steps per rollout per env (2048 * 0.002s = ~4 sec)
    num_minibatches = 32,          # Number of minibatches for each training epoch
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.99,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.95,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [5]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [6]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-1"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    yaw_penalty_coef=0.0
)

In [7]:
# Choo choo train
result = train(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-1__42__1778805721
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-1
Episode start: pitch=0.0016, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0009, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0021, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0004, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0003, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0026, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0002, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0020, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch

In [8]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Episode start: pitch=-0.0018, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0005, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0009, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0022, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0006, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0003, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0018, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0012, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0021, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0000, qpos=[0.     0.     0.

## Phase 2: Penalize position and rotation

We start with the agent (actor and critic networks) trained in phase 1. We then update the existing environments to apply a penalty for any motion off the origin as well as rotation around the Z axis (yaw). We then train the existing agent further in this new environment.

In [9]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-2"

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.yaw_penalty_coef = 0.1

In [10]:
import mujoco
for i, env_stat_wrapper in enumerate(envs.envs):
    env = env_stat_wrapper.env
    print(f"env {i}: left_wheel_dof={env._left_wheel_dof_idx}, right_wheel_dof={env._right_wheel_dof_idx}")

    env = envs.envs[0].env
print(f"nv (total DOFs): {env.model.nv}")
print(f"njnt (total joints): {env.model.njnt}")
for j in range(env.model.njnt):
    name = mujoco.mj_id2name(env.model, mujoco.mjtObj.mjOBJ_JOINT, j)
    dof = env.model.jnt_dofadr[j]
    print(f"  joint {j}: name={name}, dofadr={dof}")

env 0: left_wheel_dof=6, right_wheel_dof=7
env 1: left_wheel_dof=6, right_wheel_dof=7
env 2: left_wheel_dof=6, right_wheel_dof=7
env 3: left_wheel_dof=6, right_wheel_dof=7
nv (total DOFs): 8
njnt (total joints): 3
  joint 0: name=root, dofadr=0
  joint 1: name=wheel_left_joint, dofadr=6
  joint 2: name=wheel_right_joint, dofadr=7


In [11]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-2__42__1778806957
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-2
Episode start: pitch=0.0016, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0009, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0021, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0004, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0002, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0014, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0015, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0026, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch

In [12]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Episode start: pitch=-0.0024, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0026, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0027, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0011, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0027, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0019, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0011, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0018, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0017, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Mean return: 849.27
Exported charts_metrics.csv (5 

## Phase 3: Observation noise and action delay

With the robot staying upright and mostly in place, we add the additional difficulty of randomizing the observations (injecting noise) and introducing a random 0..1 timestep delay of how long it takes the chosen action to take effect.

The noisy observations model how real IMUs work. You will often find some noise in their readings, and it can help the robot deal with biases from an imperfectly calibrated IMU.

The action delay models the time it takes for the robot to respond to a chosen action. In simulation, a chosen action takes effect on the very next `step()` call. In real hardware, it could take up to a few milliseconds, thanks to extra code, communication bus latency (e.g. I2C), and motor lag.

In [13]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-3"

# Create data randomization config to add obs noise and action delay
dr = DomainRandomConfig(
    pitch_noise_std_dev=0.01,       # Inject noise into pitch observation
    pitch_rate_noise_std_dev=0.01,  # Inject noise into pitch rate observation
    wheel_vel_noise_std_dev=0.01,    # Inject noise into wheel velocity observation
    action_delay_steps=1,           # 1 step (5ms) delay
    action_delay_random=True,       # Vary 0-1 steps each episode
)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [14]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-3__42__1778808152
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-3
Episode start: pitch=0.0016, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0009, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0021, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0004, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0028, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0024, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0005, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0024, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch

In [15]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Episode start: pitch=-0.0000, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0019, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0010, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0010, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0017, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0009, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0005, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=-0.0021, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0019, qpos=[0.     0.     0.0225], mass=0.1270, friction=0.8000, left_gear=0.0500
Episode start: pitch=0.0012, qpos=[0.     0.     0.

## Phase 4: Full domain randomization

Once the bot has gotten used to some variation (with observation noise and random action delays), we'll enable the rest of the domain randomization variables here, which includes adding motor noise, motor torque variance, a probability of a random push each step, mass variation, and friction variation.

In [16]:
# Update experiment name
ppo_config.exp_name = "balance-bot-phase-4"

# Add full DR: motor noise, gain, friction, mass, pushes
dr.motor_noise_scale = 0.02             # +/-2% motor noise
dr.motor_gain_range = (0.7, 1.0)        # 70% to 100% motor torque each episode
dr.push_prob = 0.01                     # 1% chance of push per step
dr.push_force_max_n = 0.5               # Gentle pushes
dr.mass_scale_range = (0.8, 1.2)        # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)    # +/-50% friction variation
dr.ridge_prob = 0.000                   # Chance of random torque applied to axles
dr.ridge_torque_max_nm = 0.000          # Small nudge

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [17]:
# Choo choo train
result = train(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

Run name: BalanceBot-v0__balance-bot-phase-4__42__1778809437
TensorBoard: http://localhost:6006/#scalars&regexFilter=balance-bot-phase-4
Episode start: pitch=0.0011, qpos=[0.     0.     0.0225], mass=0.1409, friction=0.7511, left_gear=0.0479
Episode start: pitch=0.0019, qpos=[0.     0.     0.0225], mass=0.1347, friction=0.4350, left_gear=0.0353
Episode start: pitch=0.0026, qpos=[0.     0.     0.0225], mass=0.1078, friction=0.6065, left_gear=0.0411
Episode start: pitch=0.0018, qpos=[0.     0.     0.0225], mass=0.1307, friction=0.8228, left_gear=0.0465
Episode start: pitch=-0.0007, qpos=[0.     0.     0.0225], mass=0.1135, friction=0.8512, left_gear=0.0377
Episode start: pitch=0.0021, qpos=[0.     0.     0.0225], mass=0.1357, friction=0.8543, left_gear=0.0450
Episode start: pitch=-0.0013, qpos=[0.     0.     0.0225], mass=0.1022, friction=0.7275, left_gear=0.0470
Episode start: pitch=-0.0003, qpos=[0.     0.     0.0225], mass=0.1069, friction=0.5291, left_gear=0.0494
Episode start: pitch

In [18]:
# Switch to real-time timestep
eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
returns = evaluate(
    result.agent, 
    eval_episodes=3, 
    config=eval_config, 
    envs=envs)

# Print 
print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
export_tb_plots_as_csv(run_path)

Episode start: pitch=0.0021, qpos=[0.     0.     0.0225], mass=0.1512, friction=0.8704, left_gear=0.0389
Episode start: pitch=-0.0025, qpos=[0.     0.     0.0225], mass=0.1438, friction=0.6798, left_gear=0.0378
Episode start: pitch=0.0005, qpos=[0.     0.     0.0225], mass=0.1488, friction=0.4297, left_gear=0.0449
Episode start: pitch=-0.0014, qpos=[0.     0.     0.0225], mass=0.1148, friction=0.4784, left_gear=0.0384
Episode start: pitch=0.0002, qpos=[0.     0.     0.0225], mass=0.1353, friction=1.0140, left_gear=0.0476
Episode start: pitch=-0.0005, qpos=[0.     0.     0.0225], mass=0.1455, friction=0.6140, left_gear=0.0385
Episode start: pitch=-0.0007, qpos=[0.     0.     0.0225], mass=0.1326, friction=1.0331, left_gear=0.0477
Episode start: pitch=0.0018, qpos=[0.     0.     0.0225], mass=0.1082, friction=0.5468, left_gear=0.0354
Episode start: pitch=0.0020, qpos=[0.     0.     0.0225], mass=0.1196, friction=1.1717, left_gear=0.0351
Episode start: pitch=0.0000, qpos=[0.     0.     0.

## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [19]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

Closing env 0
Closing env 1
Closing env 2
Closing env 3


In [20]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

/workspace/software/rl/ppo_trainer.py:381: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0515 05:03:51.573000 11055 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0515 05:03:51.575000 11055 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0515 05:03:51.577000 11055 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Sequential([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Actor exported to ONNX: runs/BalanceBot-v0__balance-bot-phase-4__42__1778809437/actor.onnx


/opt/pyenv/versions/3.12.13/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [21]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)

Weights found:
  0.weight: shape=(16, 4), dtype=float32
  0.bias: shape=(16,), dtype=float32
  2.weight: shape=(16, 16), dtype=float32
  2.bias: shape=(16,), dtype=float32
  4.weight: shape=(2, 16), dtype=float32
  4.bias: shape=(2,), dtype=float32
C header written to runs/BalanceBot-v0__balance-bot-phase-4__42__1778809437/actor.h
  Layers:  3
  Obs:     4
  Actions: 2


PosixPath('runs/BalanceBot-v0__balance-bot-phase-4__42__1778809437/actor.h')